# Part I: Language Model Training and Comparison

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
import time
import math
import re

c:\Users\jenni\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("stanfordnlp/imdb")
train_lines = dataset["train"]["text"]

In [3]:
def tokenize(line):
    line = line.strip().lower()
    if not line:
        return []
    # Skip Wikipedia section headers like "== History =="
    if re.fullmatch(r"=+\s*.*\s*=+", line):
        return []
    # Extract words and punctuation as tokens
    return re.findall(r"\w+|[^\w\s]", line)

In [4]:
MAX_LINES = 300  # limit for speed (increase if you want a stronger model)

tokens = []
start_train1 = time.time()
for line in train_lines[:MAX_LINES]:
    toks = tokenize(line)
    if toks:
        tokens.extend(["<s>"] + toks + ["</s>"])

print("Total tokens:", len(tokens))

# Vocabulary = set of all unique tokens we observed
vocab = set(tokens)
print("Vocab size:", len(vocab))
end_train1 = time.time()

Total tokens: 86214
Vocab size: 8199


In [5]:
word_to_idx = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

In [6]:
encoded_text = [word_to_idx[word] for word in tokens]

In [7]:
data = torch.tensor(encoded_text, dtype = torch.long)

In [8]:
# LSTM
class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size) # Character embedding layer
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size) # Output layer

    def forward(self, x, hidden):
        x = self.embedding(x) # Convert character indices to embeddings
        output, hidden = self.lstm(x, hidden) # Pass through LSTM
        output = self.fc(output) # Map LSTM outputs to vocab probabilities
        return output, hidden

In [9]:
class TextDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx:idx + self.seq_len], dtype=torch.long)
        y = torch.tensor(self.data[idx + 1:idx + self.seq_len + 1], dtype=torch.long)
        return x, y

In [10]:
vocab_size = len(vocab)
seq_len = 100
batch_size = 32
embed_size = 128
hidden_size = 256
num_layers = 2
learning_rate = 0.001
num_epochs = 5

In [11]:
data = data.view(-1)
dataset = TextDataset(data, seq_len)
data_loader = DataLoader(dataset, batch_size = batch_size, shuffle = True)

In [12]:
model = LSTMLanguageModel(vocab_size, embed_size, hidden_size, num_layers)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [13]:
# Training loop
start_train = time.time()

for epoch in range(num_epochs):
    total_loss = 0
    progress_bar = tqdm(data_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for x_batch, y_batch in progress_bar:
        hidden = (torch.zeros(num_layers, x_batch.size(0), hidden_size), torch.zeros(num_layers, x_batch.size(0), hidden_size)) # Reset hidden state for each batch
        optimizer.zero_grad()
        output, hidden = model(x_batch, hidden)
        loss = criterion(output.view(-1, vocab_size), y_batch.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())
    
    print(f"Epoch {epoch+1}/{num_epochs}, Total Loss: {total_loss:.4f}")

end_train = time.time()

Epoch 1/5:   0%|          | 0/2692 [00:00<?, ?it/s]C:\Users\jenni\AppData\Local\Temp\ipykernel_3420\2358578543.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(self.data[idx:idx + self.seq_len], dtype=torch.long)
C:\Users\jenni\AppData\Local\Temp\ipykernel_3420\2358578543.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(self.data[idx + 1:idx + self.seq_len + 1], dtype=torch.long)
Epoch 1/5: 100%|██████████| 2692/2692 [09:10<00:00,  4.89it/s, loss=3.31]


Epoch 1/5, Total Loss: 12725.3501


Epoch 2/5: 100%|██████████| 2692/2692 [15:31<00:00,  2.89it/s, loss=1.49]


Epoch 2/5, Total Loss: 6256.8285


Epoch 3/5: 100%|██████████| 2692/2692 [29:35<00:00,  1.52it/s, loss=0.425]


Epoch 3/5, Total Loss: 2283.3045


Epoch 4/5: 100%|██████████| 2692/2692 [22:47<00:00,  1.97it/s, loss=0.228]


Epoch 4/5, Total Loss: 824.6543


Epoch 5/5: 100%|██████████| 2692/2692 [15:57<00:00,  2.81it/s, loss=0.163]

Epoch 5/5, Total Loss: 476.3899


In [14]:
print(loss)
print(total_loss)

tensor(0.1630, grad_fn=<NllLossBackward0>)
476.38987343013287


In [15]:
import torch.nn.functional as F

# Generate text
def generate_text(model, start_text, max_len=20):
    model.eval()
    # Convert starting text to indices and create input tensor
    input_ids = torch.tensor([word_to_idx[word] for word in start_text],
dtype=torch.long).unsqueeze(0) # Shape: (1, len(start_text))
    generated_text = list(start_text) # Store the generated words
    hidden = None # Initialize hidden state
    
    for _ in range(max_len):
        output, hidden = model(input_ids, hidden) # Forward pass
        next_word_id = output[:, -1, :].argmax(dim=-1).item() # Take the last step's output
        next_word = idx_to_word[next_word_id] # Convert to word
        generated_text.append(next_word)
        input_ids = torch.tensor([[next_word_id]], dtype=torch.long) # Update input with the predicted word
        if next_word == ".": # Stop generating at a full stop
            break
    return ' '.join(generated_text)

In [16]:
start_inf = time.time()
start_text = ["this","movie"]
print("Generated Text:", generate_text(model, start_text))
end_inf = time.time()

Generated Text: this movie , i think the nude / sexual scenes are overdone for what it is .


In [17]:
print(f"Inference time: {end_inf - start_inf:.2f}s")

Inference time: 0.04s
